# fastText Training — French OpenSubtitles Corpus

Training domain-adapted French word embeddings from the OpenSubtitles corpus with the `fasttext` Python package.

**Pipeline overview:**
1. **Preprocess** — lowercase, strip whitespace, drop empty lines, preserve accents & apostrophes
2. **Train** — skipgram model, `dim=300`, optimised for informal spoken French
3. **Diagnostics** — nearest neighbours for verlan and standard words
4. **Evaluation** — cosine similarity between verlan–standard pairs

In [11]:
import os
import time
from pathlib import Path

import numpy as np
import fasttext

# ---------------------------------------------------------------------------
# Paths — adjust DATA_DIR if the notebook is moved
# ---------------------------------------------------------------------------
DATA_DIR     = Path("data")
RAW_CORPUS   = DATA_DIR / "opensubtitles_v2024.txt"
CLEAN_CORPUS = DATA_DIR / "opensubtitles_v2024_clean.txt"
MODEL_PATH   = DATA_DIR / "ft_opensubtitles_fr_300d.bin"

## 1 — Preprocessing

In [12]:
import re

# Characters to keep: Unicode letters (incl. accented), digits, whitespace,
# and the apostrophe (both straight ' and curly \u2019) — everything else is punctuation.
_KEEP_RE = re.compile(r"[^\w\s'\u2019]", re.UNICODE)


def preprocess_corpus(src: Path, dst: Path) -> tuple[int, int]:
    """
    Read *src* line by line, apply minimal normalisation, write to *dst*.

    Normalisation:
      - lowercase
      - remove punctuation (keep letters incl. accents, digits, whitespace,
        straight apostrophe ' and curly apostrophe \u2019)
      - collapse multiple spaces produced by punctuation removal
      - strip leading/trailing whitespace
      - drop empty lines

    Returns (raw_line_count, clean_line_count).
    """
    if not src.exists():
        raise FileNotFoundError(f"Corpus not found: {src}")

    raw_count = 0
    clean_count = 0

    with src.open(encoding="utf-8") as fin, dst.open("w", encoding="utf-8") as fout:
        for line in fin:
            raw_count += 1
            cleaned = line.lower()
            cleaned = _KEEP_RE.sub(" ", cleaned)          # replace punctuation with space
            cleaned = " ".join(cleaned.split())           # collapse whitespace & strip
            if cleaned:                                   # skip empty lines
                fout.write(cleaned + "\n")
                clean_count += 1

    return raw_count, clean_count


# Run preprocessing
raw_count, clean_count = preprocess_corpus(RAW_CORPUS, CLEAN_CORPUS)

file_size_mb = CLEAN_CORPUS.stat().st_size / 1_048_576
print(f"Raw lines      : {raw_count:,}")
print(f"Clean lines    : {clean_count:,}  ({raw_count - clean_count:,} dropped)")
print(f"Clean file size: {file_size_mb:.1f} MB  →  {CLEAN_CORPUS}")

Raw lines      : 203,205,460
Clean lines    : 202,721,519  (483,941 dropped)
Clean file size: 5978.9 MB  →  data/opensubtitles_v2024_clean.txt


## 2 — Training

**Hyperparameter notes:**
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `model` | `skipgram` | Better on rare/OOV words via subword sharing — critical for verlan |
| `dim` | 300 | Matches `cc.fr.300.bin` for easy comparison |
| `ws` | 5 | Standard context window for spoken-register text |
| `epoch` | 10 | Good coverage for a corpus of this size |
| `minCount` | 3 | Keep rare verlan forms while filtering noise |
| `minn/maxn` | 3–6 | Covers French morphology and partial verlan syllable patterns |
| `neg` | 10 | Stronger negative sampling than default (5) |
| `lr` | 0.05 | fastText default; safe starting point |

In [ ]:
def train_fasttext(
    corpus: Path,
    output: Path,
    *,
    dim: int = 300,
    ws: int = 5,
    epoch: int = 10,
    min_count: int = 3,
    minn: int = 3,
    maxn: int = 6,
    neg: int = 10,
    lr: float = 0.05,
    thread: int = os.cpu_count() or 4,
) -> fasttext.FastText._FastText:
    """
    Train an unsupervised fastText skipgram model and save it to *output*.

    The function skips training if *output* already exists, allowing safe
    re-runs of the notebook.
    """
    if output.exists():
        print(f"Model already exists at {output} — loading instead of retraining.")
        return fasttext.load_model(str(output))

    print(f"Training fastText skipgram on {corpus} …")
    print(f"  dim={dim}, ws={ws}, epoch={epoch}, minCount={min_count}, "
          f"minn={minn}, maxn={maxn}, neg={neg}, lr={lr}, thread={thread}")

    t0 = time.time()
    model = fasttext.train_unsupervised(
        str(corpus),
        model="skipgram",
        dim=dim,
        ws=ws,
        epoch=epoch,
        minCount=min_count,
        minn=minn,
        maxn=maxn,
        neg=neg,
        lr=lr,
        thread=thread,
    )
    elapsed = time.time() - t0
    print(f"Training finished in {elapsed / 60:.1f} min.")

    model.save_model(str(output))
    print(f"Model saved → {output}")
    return model


# Train (or reload if already done)
ft_model = train_fasttext(CLEAN_CORPUS, MODEL_PATH)

Training fastText skipgram on data/opensubtitles_v2024_clean.txt …
  dim=300, ws=5, epoch=10, minCount=3, minn=3, maxn=6, neg=10, lr=0.05, thread=8


Read 1346M words
Number of words:  671710
Number of labels: 0
Progress:   0.6% words/sec/thread:   57474 lr:  0.049691 avg.loss:  1.993797 ETA:   8h 4m55s  0.0% words/sec/thread:   54938 lr:  0.049976 avg.loss:  2.179733 ETA:   8h30m13s  0.1% words/sec/thread:   54875 lr:  0.049965 avg.loss:  2.116849 ETA:   8h30m42s ETA:   8h28m34ss lr:  0.049946 avg.loss:  2.086473 ETA:   8h24m28s avg.loss:  2.087703 ETA:   8h24m42s  0.1% words/sec/thread:   55518 lr:  0.049933 avg.loss:  2.050612 ETA:   8h24m27sm41s  0.2% words/sec/thread:   55966 lr:  0.049887 avg.loss:  2.017564 ETA:   8h19m57s avg.loss:  2.018802 ETA:   8h17m 3s  0.3% words/sec/thread:   56678 lr:  0.049867 avg.loss:  2.016621 ETA:   8h13m29s  0.3% words/sec/thread:   56930 lr:  0.049851 avg.loss:  2.025675 ETA:   8h11m 8s  0.3% words/sec/thread:   56322 lr:  0.049836 avg.loss:  2.013215 ETA:   8h16m17s  0.4% words/sec/thread:   56501 lr:  0.049804 avg.loss:  2.003561 ETA:   8h14m24s  0.4% words/sec/thread:   56564 lr:  0.049798 

## 3 — Diagnostics: nearest neighbours & word vectors

In [4]:
def print_neighbors(model: fasttext.FastText._FastText, words: list[str], k: int = 10) -> None:
    """Print the k nearest neighbours for each word in *words*."""
    for word in words:
        neighbors = model.get_nearest_neighbors(word, k=k)
        print(f"\n  [{word}]")
        for score, neighbor in neighbors:
            print(f"    {score:.4f}  {neighbor}")


# Vocabulary size
print(f"Vocabulary size: {len(ft_model.words):,} words\n")

# Nearest neighbours for a mix of standard and verlan words
probe_words = ["femme", "homme", "parler", "meuf", "teuf"]
print("=== Nearest neighbours (k=10) ===")
print_neighbors(ft_model, probe_words)

# Example: retrieve a raw word vector
print("\n=== Word vector example ===")
vec = ft_model.get_word_vector("meuf")
print(f"  get_word_vector('meuf') → shape {vec.shape}, dtype {vec.dtype}")
print(f"  First 8 dims: {vec[:8].round(4)}")

Vocabulary size: 70,728 words

=== Nearest neighbours (k=10) ===

  [femme]
    0.9168  "femme
    0.8937  femme:
    0.8042  femme."
    0.7710  femme-
    0.7681  femmes:
    0.7493  femme!
    0.7478  femme,
    0.7295  femme?
    0.7263  femmes."
    0.7263  femme.

  [homme]
    0.8539  "homme
    0.7995  'homme
    0.7625  homme,
    0.7398  homme-
    0.7345  homme!
    0.7343  i'homme
    0.7299  homme.
    0.7277  "l'homme
    0.7242  homme..."
    0.7209  homme?

  [parler]
    0.9196  "parler
    0.8325  parler."
    0.8151  parler..
    0.7726  parler!
    0.7701  parler,
    0.7395  reparler
    0.7373  parler.
    0.7370  parler?
    0.7281  parlera
    0.7152  parle

  [meuf]
    0.8867  meuf!
    0.8492  meuf.
    0.8244  meuf,
    0.7936  meufs
    0.6941  meuh.
    0.6801  meufs,
    0.6774  meut
    0.6763  meufs!
    0.6252  meufs.
    0.6096  meg

  [teuf]
    0.8383  teuf!
    0.7943  teuf?
    0.7919  teuf.
    0.7672  teuf,
    0.6191  tej.
    0.6089  tej?
    

## 4 — Evaluation: cosine similarity between verlan–standard pairs

`get_word_vector` always returns a vector (via subword fallback), so even OOV words produce a valid embedding. The cosine score will be lower for truly rare items, but the function never raises.

In [5]:
def cosine_similarity(model: fasttext.FastText._FastText, word_a: str, word_b: str) -> float:
    """
    Compute cosine similarity between two words using their fastText vectors.

    fastText's get_word_vector() uses subword fallback, so this never raises
    for OOV words — it returns a lower-quality vector instead.
    """
    vec_a = model.get_word_vector(word_a).astype(np.float64)
    vec_b = model.get_word_vector(word_b).astype(np.float64)

    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    # Guard against zero vectors (shouldn't happen with subword fallback, but be safe)
    if norm_a == 0 or norm_b == 0:
        return 0.0

    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))


# Verlan–standard pairs to evaluate
EVAL_PAIRS = [
    ("meuf",   "femme"),    # meuf  ← femme (syllable reversal)
    ("teuf",   "fête"),     # teuf  ← fête
    ("keuf",   "flic"),     # keuf  ← flic (police slang)  — indirect link expected
    ("keuf",   "police"),   # more direct semantic link
    ("laisse", "salle"),    # verlan of salle
    ("ouf",    "fou"),      # ouf   ← fou
]

print("=== Cosine similarities (verlan ↔ standard) ===\n")
print(f"  {'Word A':<10} {'Word B':<10}  cosine")
print(f"  {'-'*10} {'-'*10}  ------")
for word_a, word_b in EVAL_PAIRS:
    score = cosine_similarity(ft_model, word_a, word_b)
    print(f"  {word_a:<10} {word_b:<10}  {score:.4f}")

=== Cosine similarities (verlan ↔ standard) ===

  Word A     Word B      cosine
  ---------- ----------  ------
  meuf       femme       0.1784
  teuf       fête        0.3784
  keuf       flic        0.2310
  keuf       police      0.2169
  laisse     salle       0.0583
  ouf        fou         0.1299


In [8]:
ft_model.get_nearest_neighbors("meuf", k=20)
# ft_model.get_nearest_neighbors("teuf", k=20)
# ft_model.get_nearest_neighbors("keuf", k=20)
# ft_model.get_nearest_neighbors("ouf", k=20)

[(0.8867022395133972, 'meuf!'),
 (0.8491581082344055, 'meuf.'),
 (0.8244269490242004, 'meuf,'),
 (0.7935534715652466, 'meufs'),
 (0.6940587162971497, 'meuh.'),
 (0.6801278591156006, 'meufs,'),
 (0.6774439215660095, 'meut'),
 (0.6762790083885193, 'meufs!'),
 (0.6252092719078064, 'meufs.'),
 (0.6096184849739075, 'meg'),
 (0.6091485023498535, 'meurt'),
 (0.6056698560714722, 'teuf'),
 (0.5987473726272583, 'megumi'),
 (0.5866890549659729, 'meurt!'),
 (0.5735301971435547, 'meute'),
 (0.5732786655426025, 'meurt,'),
 (0.56806880235672, 'meurt?'),
 (0.566296398639679, 'melo'),
 (0.5566678047180176, 'me"'),
 (0.5515633821487427, 'meurt.')]